# Welcome to the Orbit of Ops Workspace

Hello! This notebook is prepared for processing sensitive data with custom branding. Please follow the instructions below to proceed with your tasks.

In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Protecting Sensitive Data in Gen AI model responses

## Overview

Your team already has a Python function that identifies and redacts or blocks sensitive data types in Gen AI model responses. You have been asked to expand the function to block Gen AI model responses that contain [US Vehicle Identification Numbers](https://cloud.google.com/sensitive-data-protection/docs/infotypes-reference#united_states), which are sensitive data consisting of a unique 17-digit code assigned to every on-road motor vehicle in North America. 

To help you achieve this goal, complete the following subtasks by following the instructions in the cells below:

1. Run all cells in the section titled Getting started with this notebook. 

2. Expand an existing Python function in the section titled Update an existing Python function to block Gemini 3.5 Flash model responses when a US VIN has been included.

3. Generate an example text response with the following prompt to test your updated function: `Is 4Y1SL65848Z411439 an example of a US Vehicle Identification Number (VIN)?`

## Getting started with this notebook

Below are few steps to get your environment ready, including installing key Python packages and setting your environmental variables (project ID and region). 

Be sure to run each cell in consecutive order using the `Run` button (play arrow) at the top of this notebook. 

### Install necessary packages 

In [1]:
# Install Gen AI
!pip install --upgrade google-genai

# Install Cloud Data Loss Prevention
!pip install google-cloud-dlp --upgrade --user

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.0/958.0 kB 12.9 MB/s  0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.37.0
    Uninstalling google-auth-2.37.0:
      Successfully uninstalled google-auth-2.37.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [google-auth]  WARNING: Failed to remove contents in a temporary directory '/opt/micromamba/lib/python3.12/site-packages/google/~uth'.
  You can safely remove it manually.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [google-auth]  WARNING: Failed to remove contents in a temporary directory '/opt/micromamba/lib/python3.12/site-packages/google/~auth2'.
  You can safely remove it manually.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [google-auth]  WARNING: Failed to remove contents in a temporary directory '/opt/micromamba/lib/python3.12/site-packages/~oogle_auth-2.37.0.dist-info'.
  You can safely remove it manually.
  Attempting uninstall: google-genai
    Found existing installation

### Restart current runtime

To use the newly installed packages in this Jupyter runtime, you must restart the runtime. You can do this by running the cell below, which will restart the current kernel.

In [2]:
# Restart kernel after installs so that your environment can access the new packages
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

<div class="alert alert-block alert-warning">
<b><p>⚠️ The kernel is going to restart. Please wait until it is finished before continuing to the next step. ⚠️</p> When prompted, click OK to continue. </b>
</div>

### Set your project ID and region

In [ ]:
# Get the Project ID
PROJECT_ID_LIST = !gcloud config get project
PROJECT_ID = str(PROJECT_ID_LIST[0]).strip() if PROJECT_ID_LIST else ""
print(f"Project ID: {PROJECT_ID}")

# Override the single region to use the multi-region endpoint required for 3.5 Flash
LOCATION = "us" 

Project ID: qwiklabs-gcp-02-a27a1df0fe3b
Location: us-east4


### Import Gemini 3.5 Flash model

In [ ]:
# Create the API client
from google import genai


client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)

# Use the exact model assigned by the lab
model = "gemini-3.5-flash"

## Update an existing Python function to block Gemini 3.5 Flash model responses when a US VIN has been included

In this section, you revise an existing Python function to block output for [US Vehicle Identification Numbers (last entry for United States infoTypes)](https://cloud.google.com/sensitive-data-protection/docs/infotypes-reference#united_states).

In the code block below for the function, __modify the code lines after `# Add conditional return to block responses containing US Vehicle Identification Numbers (VIN)`__ to block model responses containing this infoType.

Be sure to run the cell with your final Python function code before you move onto the next cells to test the updated function.

In [30]:
# Redefine original function to inspect and deidentify output with Sensitive Data Protection
import google.cloud.dlp  
from typing import List 

def deidentify_with_replace_infotype(
    project: str, item: str, info_types: List[str]
) -> None:
    """Uses the Data Loss Prevention API to deidentify sensitive data in a
    string by replacing it with the info type.
    Args:
        project: The Google Cloud project id to use as a parent resource.
        item: The string to deidentify (will be treated as text).
        info_types: A list of strings representing info types to look for.
            A full list of info type categories can be fetched from the API.
    Returns:
        None; the response from the API is printed to the terminal.
    """

    # Instantiate a client
    dlp = google.cloud.dlp_v2.DlpServiceClient()

    # Convert the project id into a full resource id.
    parent = f"projects/{PROJECT_ID}"

    # Construct inspect configuration dictionary
    inspect_config = {"info_types": [{"name": info_type} for info_type in info_types]}

    # Construct deidentify configuration dictionary
    deidentify_config = {
        "info_type_transformations": {
            "transformations": [
                {"primitive_transformation": {"replace_with_info_type_config": {}}}
            ]
        }
    }

    # Call the API for deidentify
    response = dlp.deidentify_content(
        request={
            "parent": parent,
            "deidentify_config": deidentify_config,
            "inspect_config": inspect_config,
            "item": {"value": item},
        }
    )

    return_payload = response.item.value
    
    # Add conditional return to block responses containing US Vehicle Identification Numbers (VIN)
    info_types = ["DOCUMENT_TYPE/R&D/SOURCE_CODE"]
    inspect_config = {"info_types": [{"name": info_type} for info_type in info_types]}

    response = dlp.inspect_content(
        request={
            "parent": parent,
            "inspect_config": inspect_config,
            "item": {"value": item},
        }
    )

    if response.result.findings:
        for finding in response.result.findings:
            if finding.info_type.name == "DOCUMENT_TYPE/R&D/SOURCE_CODE":
                return_payload = '[Blocked due to category: Source Code]'
                
    # Print results
    print(return_payload)

## Generate an example with VIN using Gemini 3.5 Flash model and block results

In the code blocks below, generate an example text response containing a US Vehicle Identification Number (VIN) using the following prompt:

`Is 4Y1SL65848Z411439 an example of a US Vehicle Identification Number (VIN)?`

When generating the response, be sure to set the temperature to 0, so that the highest probability results are returned for the progress check below.

Then, write and execute the appropriate code lines to block responses containing US Vehicle Identification Numbers (VIN).

In [31]:
# Create prompt that generates an example response with US Vehicle Identification Number (VIN)
prompt = "Is 4Y1SL65848Z411439 an example of a US Vehicle Identification Number (VIN)?"

# Run model with prompt using the correct GenAI SDK syntax
response_vin = client.models.generate_content(
    model=model,
    contents=prompt
)

# Print response without blocking it (VIN provided)
print(response_vin.text)

# Block model response that includes US Vehicle Identification Number (VIN)
deidentify_with_replace_infotype(
    project=PROJECT_ID,
    item=response_vin.text,
    info_types=["US_VEHICLE_IDENTIFICATION_NUMBER"]
)

Yes, **4Y1SL65848Z411439** is a valid example of a United States (and international standard) Vehicle Identification Number (VIN). 

Here is a breakdown of why this is a valid 17-character VIN:

1. **Length:** It is exactly 17 characters long, which is the standard length for all VINs since 1981.
2. **Invalid Characters:** It does not contain the letters **I**, **O**, or **Q**, which are prohibited in all VINs to avoid confusion with the numbers 1 and 0.
3. **World Manufacturer Identifier (WMI - Characters 1-3):** 
   * **`4Y1`** indicates a vehicle manufactured in the **United States** (specifically by Subaru of Indiana Automotive).
4. **Model Year (Character 10):** 
   * **`8`** represents the model year **2008**.
5. **Assembly Plant (Character 11):** 
   * **`Z`** represents the Lafayette, Indiana assembly plant.
6. **Sequential Production Number (Characters 12-17):** 
   * **`411439`** is the unique serial number for the specific vehicle.

### What kind of vehicle is this?
Based on

In [32]:
def deidentify_with_replace_infotype(
    project: str, item: str, info_types: List[str]
) -> None:
    """Uses the Data Loss Prevention API to deidentify sensitive data in a
    string by replacing it with the info type.
    Args:
        project: The Google Cloud project id to use as a parent resource.
        item: The string to deidentify (will be treated as text).
        info_types: A list of strings representing info types to look for.
            A full list of info type categories can be fetched from the API.
    Returns:
        None; the response from the API is printed to the terminal.
    """

    # Instantiate a client
    dlp = google.cloud.dlp_v2.DlpServiceClient()

    # Convert the project id into a full resource id.
    parent = f"projects/{PROJECT_ID}"

    # Construct inspect configuration dictionary
    inspect_config = {"info_types": [{"name": info_type} for info_type in info_types]}

    # Construct deidentify configuration dictionary
    deidentify_config = {
        "info_type_transformations": {
            "transformations": [
                {"primitive_transformation": {"replace_with_info_type_config": {}}}
            ]
        }
    }

    # Call the API for deidentify
    response = dlp.deidentify_content(
        request={
            "parent": parent,
            "deidentify_config": deidentify_config,
            "inspect_config": inspect_config,
            "item": {"value": item},
        }
    )

    return_payload = response.item.value

    # Add conditional return to block responses containing US Vehicle Identification Numbers (VIN)
    vin_info_types = ["US_VEHICLE_IDENTIFICATION_NUMBER"]
    inspect_config = {"info_types": [{"name": info_type} for info_type in vin_info_types]}

    response = dlp.inspect_content(
        request={
            "parent": parent,
            "inspect_config": inspect_config,
            "item": {"value": item},
        }
    )

    if response.result.findings:
        for finding in response.result.findings:
            if finding.info_type.name == "US_VEHICLE_IDENTIFICATION_NUMBER":
                return_payload = '[Blocked due to category: US Vehicle Identification Number (VIN)]'

    # Print results
    print(return_payload)

In [40]:
import base64
from IPython.display import display, HTML

# 1. Read the local image file and encode it
with open("logo.png", "rb") as image_file:
    encoded_string = base64.b64encode(image_file.read()).decode('utf-8')

# 2. Create the data URI for the HTML src attribute
b64_image_data = f"data:image/png;base64,{encoded_string}"

# 3. Inject it directly into the HTML
display(HTML(f'''
<div style="text-align: center; padding: 25px; background-color: #ffffff; border: 2px solid #1a73e8; border-radius: 12px; margin-bottom: 25px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
    <img src="{b64_image_data}" width="250" />
    <h1 style="color: #1a73e8; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin-top: 15px;">Orbit of Ops</h1>
    <p style="color: #5f6368; font-size: 1.1em;"><i>Optimizing Operations via Intelligence</i></p>
    <hr style="width: 50%; border: 0; border-top: 1px solid #eee; margin: 15px auto;">
    <a href="https://orbitofops.com" style="color: #1a73e8; text-decoration: none; font-weight: bold;">orbitofops.com</a>
</div>
'''))